In [ ]:


from __future__ import annotations
import datetime as dt
import json, math, os, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import copy


import numpy as np
import pandas as pd
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    
    from torch.nn import TransformerEncoder, TransformerEncoderLayer
except ImportError:
    print("Warning: PyTorch not installed. The main execution block will be skipped.")
    torch = None
    nn = None
    optim = None
    TransformerEncoder = None
    TransformerEncoderLayer = None


from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import t


pm = None
ETSModel = None
Holt = None
SimpleExpSmoothing = None
ConvergenceWarning = Warning
ValueWarning = Warning

try:
    
    from statsmodels.tsa.exponential_smoothing.ets import ETSModel
    from statsmodels.tsa.api import Holt, SimpleExpSmoothing
    from statsmodels.tools.sm_exceptions import ConvergenceWarning, ValueWarning
except ImportError:
    print("Warning: statsmodels not fully installed or outdated. ETS/Smoothing baselines may be skipped.")

try:
    import pmdarima as pm


except (ImportError, ValueError) as e:
    print(f"\nWarning: Failed to import pmdarima. Auto-ARIMA baseline will be skipped.")
    print(f"Reason: {e}")
    if "numpy.dtype size changed" in str(e):
        print("\nACTION REQUIRED: This is due to NumPy binary incompatibility.")
        print("To fix, run in your terminal: pip install --upgrade numpy && pip install --upgrade --force-reinstall scipy statsmodels pmdarima\n")
    pm = None



mrmr = None
try:
    import mrmr
except ImportError:
    print("Warning: mrmr-selection not installed. Multivariate mode will fail if used.")
    mrmr = None

try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import SuccessiveHalvingPruner
except ImportError:
    print("Warning: Optuna not installed. The main execution block will be skipped.")
    optuna = None


ROOT = Path("/yourpath")

BASE_OUT = GDRIVE_ROOT / "36_forecast_outputs_transformer_rigorous"
BASE_OUT.mkdir(parents=True, exist_ok=True)
STORAGE = f"sqlite:///{BASE_OUT / 'transformer_rigorous_optuna_study.db'}"
MODEL_TYPE = "Transformer" 


FORECAST_HORIZON = 36 
SEASONAL_PERIOD = 12


OPTUNA_N_TRIALS = 50
MAX_EPOCHS_OPTUNA = 30
PATIENCE_ES = 5 
N_CV_SPLITS = 5 


MAX_EPOCHS_FINAL = 150 
WEIGHT_DECAY = 1e-4
RANDOM_STATE = 42


MC_SAMPLES_FOR_CI = 100
EPSILON = 1e-8


if torch:
    torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device("cuda" if torch and torch.cuda.is_available() else "cpu")
print("Using device:", device)


try:
    if torch:
        torch.backends.cudnn.benchmark = True
        if torch.cuda.is_available():
            if hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
                 torch.backends.cuda.matmul.allow_tf32 = True
            if hasattr(torch, 'set_float32_matmul_precision'):
                torch.set_float32_matmul_precision("medium")
except Exception:
    pass



if nn:
    class PositionalEncoding(nn.Module):
        """
        Standard sinusoidal positional encoding (Vaswani et al., 2017).
        Adapted for batch_first=True.
        """
        def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 500):
            super().__init__()
            self.dropout = nn.Dropout(p=dropout)

            position = torch.arange(max_len).unsqueeze(1).float()
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe = torch.zeros(max_len, d_model)
            pe[:, 0::2] = torch.sin(position * div_term)

            
            if d_model % 2 == 1:
                pe[:, 1::2] = torch.cos(position * div_term[:-1])
            else:
                pe[:, 1::2] = torch.cos(position * div_term)

            
            pe = pe.unsqueeze(0)
            self.register_buffer('pe', pe)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            
            x = x + self.pe[:, :x.size(1)]
            return self.dropout(x)

    class TransformerForecaster(nn.Module):
        """
        Encoder-only Transformer model for Direct Multi-Step (DMS) forecasting.
        """
        def __init__(self, input_size: int, d_model: int, nhead: int,
                     num_layers: int, dim_feedforward: int, output_size: int,
                     dropout: float, seq_len: int):
            super().__init__()
            self.d_model = d_model

            
            self.input_projection = nn.Linear(input_size, d_model)
            self.pos_encoder = PositionalEncoding(d_model, dropout, max_len=seq_len)

            
            encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True, activation="relu")
            self.transformer_encoder = TransformerEncoder(encoder_layer, num_layers)

            
            self.decoder = nn.Linear(d_model * seq_len, output_size)
            self.init_weights()

        def init_weights(self):
            initrange = 0.1
            self.input_projection.weight.data.uniform_(-initrange, initrange)
            self.decoder.bias.data.zero_()
            self.decoder.weight.data.uniform_(-initrange, initrange)

        def forward(self, src):
            

            
            src = self.input_projection(src) * math.sqrt(self.d_model)
            src = self.pos_encoder(src)

            
            output = self.transformer_encoder(src)

            
            output = output.reshape(output.size(0), -1)

            
            output = self.decoder(output)
            
            return output


def build_model(hp: Dict, n_vars: int, n_lags: int) -> nn.Module:
    if not nn: return None

    
    d_model = int(hp["d_model"])
    nhead = int(hp["nhead"])

    
    dim_feedforward = d_model * int(hp["dim_feedforward_factor"])


    
    if d_model % nhead != 0:
        
        print(f"Warning: Invalid config d_model={d_model}, nhead={nhead}. Skipping build.")
        return None

    mdl = TransformerForecaster(
        input_size=n_vars,
        d_model=d_model,
        nhead=nhead,
        num_layers=int(hp["num_layers"]),
        dim_feedforward=dim_feedforward,
        output_size=FORECAST_HORIZON,
        dropout=float(hp["dropout_rate"]),
        seq_len=n_lags 
    ).to(device)

    try:
        
        if hasattr(torch, 'compile'):
             return torch.compile(mdl, mode="reduce-overhead")
        return mdl
    except Exception:
        return mdl




def smape_loss_np(y_true, y_pred, eps: float = EPSILON) -> float:
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    num = np.abs(y_true - y_pred)
    den = (np.abs(y_true) + np.abs(y_pred) + eps) / 2.0
    return np.mean(num / den)

def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray, insample: Optional[np.ndarray] = None):
    if y_true.ndim == 1 or (y_true.ndim > 1 and y_true.shape[1] == 1):
        y_true = y_true.ravel(); y_pred = y_pred.ravel()

    smape = smape_loss_np(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + EPSILON))) * 100.0
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mase = rmsse = np.nan

    if insample is not None and len(insample) > SEASONAL_PERIOD:
        d = np.mean(np.abs(np.diff(insample, n=SEASONAL_PERIOD)))
        mae = np.mean(np.abs(y_true - y_pred)); mse = np.mean((y_true - y_pred) ** 2)
        mase = mae / (d + EPSILON)
        denom_rmsse = np.mean((np.diff(insample, n=SEASONAL_PERIOD))**2)
        rmsse = np.sqrt(mse / (denom_rmsse + EPSILON))

    return {"SMAPE": smape, "MAPE": mape, "RMSE": rmse, "MASE": mase, "RMSSE": rmsse}

def diebold_mariano_test(actuals: np.ndarray, pred1: np.ndarray, pred2: np.ndarray, horizon: int):
    actuals = actuals.ravel(); pred1 = pred1.ravel(); pred2 = pred2.ravel()
    if len(actuals) != len(pred1) or len(actuals) != len(pred2): return np.nan, np.nan
    N = len(actuals)
    if N == 0: return np.nan, np.nan

    loss1 = (actuals - pred1)**2; loss2 = (actuals - pred2)**2
    d = loss1 - loss2; d_mean = np.mean(d)

    q = horizon; gamma = np.zeros(q + 1)
    for k in range(q + 1):
        if k == 0:
            gamma[k] = np.var(d, ddof=0)
        else:
            try:
                cov = np.cov(d[k:], d[:N-k], ddof=0)
                if cov.ndim == 2: gamma[k] = cov[0, 1]
                else: gamma[k] = 0
            except Exception: gamma[k] = 0

    V = gamma[0] + 2 * np.sum([(1 - k/(q+1)) * gamma[k] for k in range(1, q+1)])

    if V <= 1e-9:
        if abs(d_mean) < 1e-9: return 0, 1.0
        else: return np.inf * np.sign(d_mean), 0.0

    DM_stat = d_mean / np.sqrt(V / N)
    p_value = 2 * (1 - t.cdf(np.abs(DM_stat), df=N-1))
    return DM_stat, p_value


def fit_smoother(series, method, alpha=None, beta=None):
    if method == "NS": return None
    if SimpleExpSmoothing is None or Holt is None:
        if method in ["ES", "DES"]: print("Warning: Smoothing methods requested but statsmodels components are missing.")
        return None
    try:
        if method == "ES":
            return SimpleExpSmoothing(series, initialization_method="estimated").fit(smoothing_level=alpha, optimized=False)
        if method == "DES":
            return Holt(series, initialization_method="estimated").fit(smoothing_level=alpha, smoothing_trend=beta, optimized=False)
    except Exception: return None
    raise ValueError(f"Unknown smoothing method: {method}")

def apply_fitted_smoother(fitter, df: pd.DataFrame, target: str) -> pd.DataFrame:
    if fitter is None: return df
    n = len(df); fitted_vals = fitter.fittedvalues
    if len(fitted_vals) == 0: return df

    if len(fitted_vals) == n: values = fitted_vals
    elif len(fitted_vals) < n: values = pd.concat([df[target].iloc[:n-len(fitted_vals)], fitted_vals])
    else: values = fitted_vals[:n]

    out = df.copy()
    if len(values) == n: out[target] = values.values
    else: return df

    out[target].bfill(inplace=True)
    return out


def create_sequences_dms(df_in: pd.DataFrame, target: str, features: List[str], num_lags: int, horizon: int) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    cols = [target] + features; data = df_in[cols].values; N = data.shape[0]
    X, Y = [], []
    for i in range(N - num_lags - horizon + 1):
        x_seq = data[i : i + num_lags]
        y_seq = data[i + num_lags : i + num_lags + horizon, 0]
        X.append(x_seq); Y.append(y_seq)
    return np.array(X), np.array(Y), cols

def select_mrmr_features(df_window: pd.DataFrame, target: str, k: int, num_lags: int) -> List[str]:
    if k == 0 or mrmr is None: return []
    lagged = []; potential_features = [c for c in df_window.columns if c != target]
    if not potential_features: return []

    for col in [target] + potential_features:
        for l in range(1, num_lags + 1):
            lagged.append(df_window[col].shift(l).rename(f"{col}_lag_{l}"))

    df_lagged = pd.concat([df_window[[target]], *lagged], axis=1).dropna()
    if df_lagged.empty: return []

    X_cols = [c for c in df_lagged.columns if "_lag_" in c and not c.startswith(f"{target}_lag_")]
    if not X_cols: return []

    X = df_lagged[X_cols].ffill().fillna(0); y = df_lagged[target].values.ravel()
    K_eff = min(k, len(X_cols))

    try:
        selected_lagged = mrmr.mrmr_regression(X=X, y=y, K=K_eff)
        selected_base = list({s.split("_lag_")[0] for s in selected_lagged})
        return selected_base
    except Exception as e:
        print(f"Warning: mRMR failed: {e}. Falling back to using no exogenous features.")
        return []


def mc_pred(model: nn.Module, inp: torch.Tensor, mc_samples: int):
    """Generates Monte Carlo predictions if dropout is active."""
    
    if mc_samples == 1:
        model.eval()
        with torch.no_grad():
            
            return model(inp).cpu().numpy()

    
    model.train()
    with torch.no_grad():
        
        inp_mc = inp.repeat(mc_samples, 1, 1)
        
        preds = model(inp_mc).cpu().numpy()
    return preds



def make_objective(tgt: str, train_val_df: pd.DataFrame,
                   k: int, mode: str):
    
    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS, test_size=FORECAST_HORIZON)

    def objective(trial: optuna.Trial):

        
        d_model = trial.suggest_categorical("d_model", [64, 128, 256, 512])

        
        possible_heads = [h for h in [4, 8, 16] if d_model >= h and d_model % h == 0]
        if not possible_heads:
             
             return float("inf")

        nhead = trial.suggest_categorical("nhead", possible_heads)

        
        dim_feedforward_factor = trial.suggest_categorical("dim_feedforward_factor", [2, 4])

        hp = {
            
            "d_model": d_model,
            "nhead": nhead,
            "num_layers": trial.suggest_int("num_layers", 1, 3), # (analogy: RNN num_layers)
            "dim_feedforward_factor": dim_feedforward_factor,
            "dropout_rate": trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1),

            
            "lr": trial.suggest_categorical("lr", [1e-4, 5e-4, 1e-3]),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),

            
            "n_lags": trial.suggest_int("n_lags", SEASONAL_PERIOD, SEASONAL_PERIOD * 4, step=SEASONAL_PERIOD),

            
            "smoothing_method": trial.suggest_categorical("smoothing_method", ["NS", "ES", "DES"]),
            "smoothing_alpha": trial.suggest_float("smoothing_alpha", 0.1, 0.5),
            "smoothing_beta": trial.suggest_float("smoothing_beta", 0.05, 0.3),
        }

        n_lags = hp["n_lags"]
        smapes_cv = []

        
        for tr_idx, val_idx in tscv.split(train_val_df):
            
            if len(tr_idx) < n_lags + FORECAST_HORIZON:
                continue

            df_tr = train_val_df.iloc[tr_idx]

            
            smoother = fit_smoother(df_tr[tgt], hp["smoothing_method"], hp["smoothing_alpha"], hp["smoothing_beta"])
            df_tr_sm = apply_fitted_smoother(smoother, df_tr, tgt)

            
            if mode == "MULTI":
                base_feats = select_mrmr_features(df_tr_sm, tgt, k, num_lags=SEASONAL_PERIOD)
            else:
                base_feats = []

            n_vars = len([tgt, *base_feats])

            
            X_tr, Y_tr, _ = create_sequences_dms(df_tr_sm, tgt, base_feats, n_lags, FORECAST_HORIZON)

            
            X_va_input = df_tr_sm[[tgt] + base_feats].iloc[-n_lags:].values.reshape(1, n_lags, n_vars)
            Y_va_target = train_val_df.iloc[val_idx][tgt].values.reshape(1, FORECAST_HORIZON)

            if X_tr.shape[0] == 0:
                continue

            
            X_tr_reshaped = X_tr.reshape(-1, n_vars); Y_tr_reshaped = Y_tr.reshape(-1, 1)
            sc_X = MinMaxScaler().fit(X_tr_reshaped); sc_y = MinMaxScaler().fit(Y_tr_reshaped)

            X_tr_scaled = sc_X.transform(X_tr.reshape(-1, n_vars)).reshape(X_tr.shape)
            Y_tr_scaled = sc_y.transform(Y_tr.reshape(-1, 1)).reshape(Y_tr.shape)
            X_va_scaled = sc_X.transform(X_va_input.reshape(-1, n_vars)).reshape(X_va_input.shape)

            
            Xtr_t = torch.tensor(X_tr_scaled, dtype=torch.float32)
            Ytr_t = torch.tensor(Y_tr_scaled, dtype=torch.float32)
            Xva_t = torch.tensor(X_va_scaled, dtype=torch.float32).to(device)

            dl = torch.utils.data.DataLoader(
                torch.utils.data.TensorDataset(Xtr_t, Ytr_t),
                batch_size=hp["batch_size"], shuffle=True)

            
            mdl = build_model(hp, n_vars, n_lags)

            if mdl is None:
                 
                 return float("inf")


            crit = nn.L1Loss()
            opt = optim.AdamW(
                mdl.parameters(), lr=hp["lr"],
                weight_decay=WEIGHT_DECAY)

            best_fold_smape = float("inf")
            no_imp = 0

            for ep in range(MAX_EPOCHS_OPTUNA):
                mdl.train()
                for xb, yb in dl:
                    xb, yb = xb.to(device), yb.to(device)
                    opt.zero_grad(set_to_none=True)
                    
                    with torch.autocast(device.type, enabled=(device.type == "cuda")):
                        preds = mdl(xb)
                        
                        loss = crit(preds, yb)

                    
                    if torch.isnan(loss):
                        return float("inf")

                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
                    opt.step()

                
                mdl.eval()
                with torch.no_grad():
                    
                    preds_scaled = mdl(Xva_t).cpu().numpy() 
                    
                    preds_inv = sc_y.inverse_transform(preds_scaled.reshape(-1, 1)).reshape(1, FORECAST_HORIZON)

                    
                    s = smape_loss_np(Y_va_target, preds_inv)

                
                trial.report(s, ep)
                if trial.should_prune():
                    raise optuna.TrialPruned()

                if s < best_fold_smape - 1e-4:
                    best_fold_smape = s
                    no_imp = 0
                else:
                    no_imp += 1

                if no_imp >= PATIENCE_ES:
                    break

            smapes_cv.append(best_fold_smape)

        if not smapes_cv:
            return float("inf")

        
        return float(np.mean(smapes_cv))

    return objective



def train_final_model(hp: Dict, df_train: pd.DataFrame, tgt: str, base_feats: List[str], use_early_stopping: bool = True):
    """
    Trains the Transformer model with optimized hyperparameters.
    """
    n_lags = hp["n_lags"]
    n_vars = len([tgt, *base_feats])

    
    smoother = fit_smoother(df_train[tgt], hp["smoothing_method"], hp["smoothing_alpha"], hp["smoothing_beta"])
    df_train_sm = apply_fitted_smoother(smoother, df_train, tgt)

    
    X, Y, _ = create_sequences_dms(df_train_sm, tgt, base_feats, n_lags, FORECAST_HORIZON)

    if X.shape[0] == 0:
        return None, None, None

    
    if use_early_stopping and X.shape[0] > 20:
         split_idx = int(X.shape[0] * 0.8)
         X_tr, Y_tr = X[:split_idx], Y[:split_idx]
         X_va, Y_va = X[split_idx:], Y[split_idx:]
    else:
        X_tr, Y_tr = X, Y
        X_va, Y_va = None, None

    
    X_tr_reshaped = X_tr.reshape(-1, n_vars); Y_tr_reshaped = Y_tr.reshape(-1, 1)
    sc_X = MinMaxScaler().fit(X_tr_reshaped); sc_y = MinMaxScaler().fit(Y_tr_reshaped)

    X_tr_scaled = sc_X.transform(X_tr.reshape(-1, n_vars)).reshape(X_tr.shape)
    Y_tr_scaled = sc_y.transform(Y_tr.reshape(-1, 1)).reshape(Y_tr.shape)

    Xtr_t = torch.tensor(X_tr_scaled, dtype=torch.float32)
    Ytr_t = torch.tensor(Y_tr_scaled, dtype=torch.float32)

    if X_va is not None:
        X_va_scaled = sc_X.transform(X_va.reshape(-1, n_vars)).reshape(X_va.shape)
        Y_va_scaled = sc_y.transform(Y_va.reshape(-1, 1)).reshape(Y_va.shape)
        Xva_t = torch.tensor(X_va_scaled, dtype=torch.float32).to(device)
        Yva_t = torch.tensor(Y_va_scaled, dtype=torch.float32).to(device)
    else:
        Xva_t, Yva_t = None, None

    dl = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr_t, Ytr_t),
        batch_size=hp["batch_size"], shuffle=True)

    
    mdl = build_model(hp, n_vars, n_lags)
    if mdl is None: return None, None, None

    crit = nn.L1Loss()
    opt = optim.AdamW(
        mdl.parameters(), lr=hp["lr"],
        weight_decay=WEIGHT_DECAY)

    
    best_val_loss = float("inf")
    no_imp = 0
    best_model_state = None

    if Xva_t is None:
         best_model_state = copy.deepcopy(mdl.state_dict())

    for ep in range(MAX_EPOCHS_FINAL):
        mdl.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, enabled=(device.type == "cuda")):
                preds = mdl(xb)
                loss = crit(preds, yb)

            if torch.isnan(loss):
                return None, None, None

            loss.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step()

        
        if Xva_t is not None:
            mdl.eval()
            with torch.no_grad():
                 with torch.autocast(device.type, enabled=(device.type == "cuda")):
                    val_preds = mdl(Xva_t)
                    val_loss = crit(val_preds, Yva_t).item()

            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                no_imp = 0
                best_model_state = copy.deepcopy(mdl.state_dict())
            else:
                no_imp += 1

            if no_imp >= PATIENCE_ES:
                break

    
    if best_model_state:
         mdl.load_state_dict(best_model_state)

    return mdl, sc_X, sc_y


def run_hpo_pipeline(df_dev: pd.DataFrame, targets: List[str], k: int, mode: str) -> Dict:
    """
    Runs the Hyperparameter Optimization pipeline.
    """
    best_cfgs = {}
    print(f"\n--- Starting HPO Pipeline (Model: {MODEL_TYPE}, Mode: {mode}) ---")

    for tgt in targets:
        print(f"\nOptimizing Target: {tgt}")

        
        study_name = f"{MODEL_TYPE}_{mode}_{tgt}_HPO_DMS"
        try:
            study = optuna.create_study(
                study_name=study_name,
                direction="minimize",
                sampler=TPESampler(seed=RANDOM_STATE),
                pruner=SuccessiveHalvingPruner(min_resource=4, reduction_factor=3),
                storage=STORAGE, load_if_exists=True)
        except Exception as e:
             print(f"Could not create/load Optuna study: {e}")
             continue

        
        if len(study.trials) < OPTUNA_N_TRIALS:
            objective_fn = make_objective(tgt, df_dev, k, mode)
            try:
                study.optimize(objective_fn,
                               n_trials=OPTUNA_N_TRIALS - len(study.trials),
                               n_jobs=1,
                               show_progress_bar=True)
            except Exception as e:
                 print(f"Optuna optimization failed for {tgt}: {e}")
                 continue


        if study.best_trial is None:
            print(f"HPO failed for {tgt}. Skipping.")
            continue

        print(f"Best CV SMAPE for {tgt}: {study.best_value:.4f}")

        best_hp = study.best_params

        
        if mode == "MULTI":
            smoother = fit_smoother(df_dev[tgt],
                                    best_hp.get("smoothing_method", "NS"),
                                    best_hp.get("smoothing_alpha", 0.5),
                                    best_hp.get("smoothing_beta", 0.1))
            df_dev_sm = apply_fitted_smoother(smoother, df_dev, tgt)
            final_feats = select_mrmr_features(df_dev_sm, tgt, k, num_lags=SEASONAL_PERIOD)
        else:
            final_feats = []

        best_cfgs[tgt] = {
            "smape_dev": study.best_value,
            "hyperparams": best_hp,
            "base_features": final_feats,
            "n_vars_per_step": len([tgt, *final_feats]),
            "n_lags": best_hp["n_lags"]
        }

    return best_cfgs



def generate_forecast_dms(
    df_hist: pd.DataFrame, target: str, cfg: Dict, horizon: int,
    mc_samples: int):
    """
    Trains the model on historical data and generates a H-step ahead forecast with CI.
    """
    if not cfg or not cfg.get("hyperparams"):
        return None

    hp = cfg["hyperparams"]
    feats = cfg["base_features"]
    n_lags = cfg["n_lags"]
    n_vars = cfg["n_vars_per_step"]

    if horizon != FORECAST_HORIZON:
         raise ValueError(f"Model architecture fixed to horizon {FORECAST_HORIZON}, cannot forecast {horizon}.")

    
    mdl, sc_X, sc_y = train_final_model(hp, df_hist, target, feats, use_early_stopping=True)

    if mdl is None:
        return None

    
    smoother = fit_smoother(df_hist[target], hp["smoothing_method"], hp["smoothing_alpha"], hp["smoothing_beta"])
    df_hist_sm = apply_fitted_smoother(smoother, df_hist, target)

    input_data = df_hist_sm[[target] + feats].iloc[-n_lags:].values

    
    input_scaled = sc_X.transform(input_data.reshape(-1, n_vars))
    
    inp_t = torch.tensor(input_scaled.reshape(1, n_lags, n_vars),
                         dtype=torch.float32, device=device)

    
    use_mc = mc_samples > 1 and hp.get("dropout_rate", 0) > 0
    samples = mc_samples if use_mc else 1

    
    preds_mc_scaled = mc_pred(mdl, inp_t, samples)

    
    preds_inv = sc_y.inverse_transform(preds_mc_scaled.reshape(-1, 1)).reshape(samples, horizon)

    
    if use_mc:
        
        forecasts = np.median(preds_inv, axis=0)
        lowers = np.percentile(preds_inv, 2.5, axis=0)
        uppers = np.percentile(preds_inv, 97.5, axis=0)
    else:
        forecasts = preds_inv[0]
        lowers = np.full(horizon, np.nan)
        uppers = np.full(horizon, np.nan)

    
    forecasts[forecasts < 0] = 0
    if use_mc:
        lowers[lowers < 0] = 0
        uppers[uppers < 0] = 0

    return pd.DataFrame({"Forecast": forecasts,
                         "Lower_CI": lowers, "Upper_CI": uppers})



def forecast_arima(series: pd.Series, horizon: int):
    if pm is None: return np.full(horizon, np.nan)
    try:
        model = pm.auto_arima(series, start_p=1, start_q=1, max_p=5, max_q=5, m=SEASONAL_PERIOD,
                              start_P=0, seasonal=True, d=None, D=1, trace=False,
                              error_action='ignore', suppress_warnings=True, stepwise=True)
        forecast = model.predict(n_periods=horizon)
        return np.asarray(forecast)
    except Exception: return np.full(horizon, np.nan)

def forecast_ets(series: pd.Series, horizon: int):
    if ETSModel is None: return np.full(horizon, np.nan)
    try:
        model = ETSModel(series, error="add", trend="add", seasonal="add",
                         damped_trend=True, seasonal_periods=SEASONAL_PERIOD)
        fit = model.fit(disp=False, optimized=True)
        forecast = fit.forecast(horizon)
        return np.asarray(forecast)
    except Exception: return np.full(horizon, np.nan)



if __name__ == "__main__":
    if not torch or not optuna:
        print("\nExecution aborted due to missing critical dependencies (PyTorch or Optuna).")
        # exit()

    
    if ConvergenceWarning: warnings.simplefilter("ignore", ConvergenceWarning)
    if ValueWarning: warnings.simplefilter("ignore", ValueWarning)
    warnings.simplefilter("ignore", UserWarning)

    if optuna:
        try: optuna.logging.set_verbosity(optuna.logging.WARNING)
        except Exception: pass

    
    env_info = {
        "timestamp_utc": dt.datetime.utcnow().isoformat() + "Z",
        "model_type": MODEL_TYPE,
        "torch": torch.__version__ if torch else "N/A",
    }
    if torch and torch.cuda.is_available():
        try: env_info["device"] = torch.cuda.get_device_name(0)
        except Exception: env_info["device"] = "CUDA (Name Unavailable)"
    else: env_info["device"] = "CPU"

    try: json.dump(env_info, open(BASE_OUT / "env.json", "w"), indent=2)
    except Exception as e: print(f"Could not write env.json: {e}")

    
    DF_PATH = ROOT / "yourdata"

    if DF_PATH.exists():
        df_raw = pd.read_excel(DF_PATH)
    else:
         print(f"Data file not found at {DF_PATH}. Proceeding with MOCK DATA.")




    date_col = next((c for c in df_raw.columns
                     if c.lower().strip() == "date"), None)
    if date_col:
        df_raw[date_col] = pd.to_datetime(df_raw[date_col])
        df_raw.sort_values(date_col, inplace=True)
        df_raw.set_index(date_col, inplace=True)

    df_raw.columns = df_raw.columns.str.lower().str.strip()

    
    for col in df_raw.columns:
        if df_raw[col].dtype == "object":
            try:
                df_raw[col] = pd.to_numeric(
                    df_raw[col].str.replace(r"[^\d.\-]", "", regex=True),
                    errors="coerce")
            except AttributeError:
                
                df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

    
    df_raw = df_raw.ffill().fillna(0)


    
    df_dev = df_raw.iloc[:-FORECAST_HORIZON].copy()
    df_test = df_raw.iloc[-FORECAST_HORIZON:].copy()

    if len(df_dev) < FORECAST_HORIZON * 3:
         print("Warning: Development set might be short relative to the horizon.")

    
    targets_all = [
        "generative ai journal article",
        "autonomous vehicles journal article",
        "digital twins journal article",
        "misinformation journal article",
        "3d printing journal article", "new programming models journal article",
        "renewable energy journal article",
        "sustainable technologies journal article", "generative agritech journal article",
        "metaverse journal article",
    ]
    targets = [t.lower().strip() for t in targets_all if t.lower().strip() in df_dev.columns]

    if not targets:
        targets = [col for col in df_raw.columns if 'journal article' in col]
        if not targets:
             raise RuntimeError("No valid targets found in data.")

    
    K_FEATURES = 10
    best_cfgs_modes = {}

    
    for MODE in ["UNI", "MULTI"]:
        if MODE == "MULTI" and (mrmr is None or len(df_raw.columns) < 2):
            print(f"Skipping {MODE} mode."); continue

        
        best_cfgs = run_hpo_pipeline(df_dev, targets, k=K_FEATURES, mode=MODE)
        best_cfgs_modes[MODE] = best_cfgs

        
        best_path = BASE_OUT / f"best_hyperparameters_{MODE}.json"
        try:
            tmp = best_path.with_suffix(".tmp")
            with open(tmp, 'w') as f:
                class NumpyEncoder(json.JSONEncoder):
                    def default(self, obj):
                        if isinstance(obj, np.integer): return int(obj)
                        elif isinstance(obj, np.floating): return float(obj)
                        elif isinstance(obj, np.ndarray): return obj.tolist()
                        return super(NumpyEncoder, self).default(obj)

                json.dump(best_cfgs, f, indent=2, cls=NumpyEncoder)
            tmp.replace(best_path)
        except Exception as e:
            print(f"Could not save HPO results for {MODE}: {e}")

    
    print("\n--- Starting Test Set (Hold-out) Evaluation ---")
    test_results = []; baseline_cache = {}

    for MODE in best_cfgs_modes.keys():
        best_cfgs = best_cfgs_modes[MODE]

        for tgt in targets:
            cfg = best_cfgs.get(tgt)
            if not cfg: continue

            
            fc_df = generate_forecast_dms(df_dev, tgt, cfg, FORECAST_HORIZON, mc_samples=1)

            if fc_df is None:
                print(f"Skipping {tgt} ({MODE}) due to model training failure."); continue

            y_true = df_test[tgt].values.ravel()
            y_pred_model = fc_df["Forecast"].values.ravel()
            insample_data = df_dev[tgt].values

            model_m = calculate_metrics(y_true, y_pred_model, insample_data)
            result_row = {
                "Target": tgt, "Mode": MODE,
                **{f"{MODEL_TYPE}_{k}": v for k, v in model_m.items()}
            }

            
            if tgt not in baseline_cache:
                print(f"Calculating Baselines for {tgt}...")
                naive_pred = np.repeat(insample_data[-1], FORECAST_HORIZON)
                if len(insample_data) >= SEASONAL_PERIOD:
                    snaive_pred = np.tile(insample_data[-SEASONAL_PERIOD:],
                                          math.ceil(FORECAST_HORIZON/SEASONAL_PERIOD))[:FORECAST_HORIZON]
                else: snaive_pred = naive_pred

                arima_pred = forecast_arima(df_dev[tgt], FORECAST_HORIZON)
                ets_pred = forecast_ets(df_dev[tgt], FORECAST_HORIZON)

                baseline_cache[tgt] = {"Naive": naive_pred, "SNaive": snaive_pred,
                                       "ARIMA": arima_pred, "ETS": ets_pred}

            
            for name, pred in baseline_cache[tgt].items():
                 if np.isnan(pred).all():
                     metrics = {"SMAPE": np.nan, "MAPE": np.nan, "RMSE": np.nan, "MASE": np.nan, "RMSSE": np.nan}
                 else:
                    metrics = calculate_metrics(y_true, pred, insample_data)
                 result_row.update({f"{name}_{k}": v for k, v in metrics.items()})

            
            arima_smape = result_row.get("ARIMA_SMAPE", np.inf)
            ets_smape = result_row.get("ETS_SMAPE", np.inf)

            if np.isnan(arima_smape): arima_smape = np.inf
            if np.isnan(ets_smape): ets_smape = np.inf

            best_stat_name = None
            if arima_smape != np.inf or ets_smape != np.inf:
                if arima_smape <= ets_smape:
                    best_stat_name = "ARIMA"; best_stat_pred = baseline_cache[tgt]["ARIMA"]
                else:
                    best_stat_name = "ETS"; best_stat_pred = baseline_cache[tgt]["ETS"]

            if best_stat_name:
                if np.isnan(y_pred_model).any() or np.isnan(best_stat_pred).any():
                    dm_stat, dm_p = np.nan, np.nan
                else:
                    dm_stat, dm_p = diebold_mariano_test(y_true, y_pred_model, best_stat_pred, FORECAST_HORIZON)

                result_row[f"DM_vs_{best_stat_name}_Stat"] = dm_stat
                result_row[f"DM_vs_{best_stat_name}_Pval"] = dm_p

            test_results.append(result_row)

    
    df_results = pd.DataFrame(test_results)

    if not df_results.empty:
        try:
            
            if len(df_results["Mode"].unique()) > 1:
                modes = list(df_results["Mode"].unique())
                df_base = df_results[df_results["Mode"] == modes[0]].drop(columns="Mode")

                for mode in modes[1:]:
                    df_mode = df_results[df_results["Mode"] == mode]
                    model_cols = [col for col in df_mode.columns if col.startswith(f"{MODEL_TYPE}_") or col.startswith("DM_") or col == "Target"]
                    df_mode = df_mode[model_cols]

                    rename_dict = {c: c.replace(f"{MODEL_TYPE}_", f"{MODEL_TYPE}_{mode}_") for c in model_cols if c.startswith(f"{MODEL_TYPE}_")}
                    rename_dm = {}
                    for c in model_cols:
                        if c.startswith("DM_"):
                             rename_dm[c] = c.replace("DM_vs_", f"DM_{mode}_vs_")

                    df_mode = df_mode.rename(columns=rename_dict, inplace=False)
                    df_mode = df_mode.rename(columns=rename_dm, inplace=False)

                    df_base = pd.merge(df_base, df_mode, on="Target", how="outer")

                df_final_results = df_base
            else:
                 df_final_results = df_results.drop(columns="Mode", errors='ignore')

            df_final_results.to_csv(BASE_OUT / "test_evaluation_summary.csv", index=False)
            print("\nTest Evaluation Summary saved.")

        except Exception as e:
            print(f"Error restructuring results table: {e}")
            df_results.to_csv(BASE_OUT / "test_evaluation_raw.csv", index=False)


    
    print("\n--- Generating Final 36-Month Future Forecasts (using full data) ---")
    df_full = pd.concat([df_dev, df_test])

    for MODE, best_cfgs in best_cfgs_modes.items():
        for tgt in targets:
            cfg = best_cfgs.get(tgt)
            if not cfg: continue

            
            fc_df = generate_forecast_dms(df_full, tgt, cfg, FORECAST_HORIZON, MC_SAMPLES_FOR_CI)

            if fc_df is None:
                print(f"Skipping future projection for {tgt} ({MODE})."); continue

            
            safe_name = "".join(c if c.isalnum() else "_" for c in tgt)
            fc_df.to_csv(BASE_OUT / f"forecast_future_36m_{MODE.lower()}_{safe_name}.csv", index=False)

    print(f"\nAll tasks finished successfully. Outputs are located at: {BASE_OUT}")


Using device: cuda

--- Starting HPO Pipeline (Model: Transformer, Mode: UNI) ---

Optimizing Target: quantum computing journal article


  0%|          | 0/50 [00:00<?, ?it/s]

W0805 09:10:01.390000 45783 site-packages/torch/_dynamo/convert_frame.py:964] [0/8] torch._dynamo hit config.recompile_limit (8)
W0805 09:10:01.390000 45783 site-packages/torch/_dynamo/convert_frame.py:964] [0/8]    function: 'forward' (/tmp/ipykernel_45783/3371854194.py:193)
W0805 09:10:01.390000 45783 site-packages/torch/_dynamo/convert_frame.py:964] [0/8]    last reason: 0/7: GLOBAL_STATE changed: grad_mode autocast 
W0805 09:10:01.390000 45783 site-packages/torch/_dynamo/convert_frame.py:964] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0805 09:10:01.390000 45783 site-packages/torch/_dynamo/convert_frame.py:964] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.


Best CV SMAPE for quantum computing journal article: 0.7015

Optimizing Target: generative ai journal article


  0%|          | 0/50 [00:00<?, ?it/s]

Best CV SMAPE for generative ai journal article: 0.7095

Optimizing Target: llm journal article


  0%|          | 0/50 [00:00<?, ?it/s]

Best CV SMAPE for llm journal article: 0.5410

Optimizing Target: autonomous vehicles journal article


  0%|          | 0/50 [00:00<?, ?it/s]